In [ ]:
import sys
print(sys.executable)


In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
import torch
import torchaudio
import numpy as np
import pandas as pd
from transformers import Wav2Vec2ForSequenceClassification, Wav2Vec2Processor
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda')

DATA_ROOT    = r'C:\Users\GHANSHYAM\Desktop\voice-clone-detector\data\raw\LA\LA'
PROTOCOL_DIR = f'{DATA_ROOT}\\ASVspoof2019_LA_cm_protocols'
TRAIN_AUDIO  = f'{DATA_ROOT}\\ASVspoof2019_LA_train\\flac'
DEV_AUDIO    = f'{DATA_ROOT}\\ASVspoof2019_LA_dev\\flac'

train_df = pd.read_csv(f'{PROTOCOL_DIR}\\ASVspoof2019.LA.cm.train.trn.txt',
    sep=' ', header=None,
    names=['speaker_id', 'file_id', 'env', 'attack_id', 'label'])

dev_df = pd.read_csv(f'{PROTOCOL_DIR}\\ASVspoof2019.LA.cm.dev.trl.txt',
    sep=' ', header=None,
    names=['speaker_id', 'file_id', 'env', 'attack_id', 'label'])

print("Libraries loaded ✅")
print(f"Train: {len(train_df)} | Dev: {len(dev_df)}")

In [ ]:
from transformers import Wav2Vec2Processor, Wav2Vec2ForSequenceClassification

print("Loading Wav2Vec2 — ~1GB download hoga pehli baar...")

processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")
model = Wav2Vec2ForSequenceClassification.from_pretrained(
    "facebook/wav2vec2-base",
    num_labels=2
)
model = model.to(device)

print(f"Model loaded ✅")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
import torchaudio

class ASVspoofDataset(Dataset):
    def __init__(self, df, audio_dir, processor, max_length=64000):
        self.df = df.reset_index(drop=True)
        self.audio_dir = audio_dir
        self.processor = processor
        self.max_length = max_length  # 4 seconds at 16kHz

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        file_path = f'{self.audio_dir}\\{row["file_id"]}.flac'

        # load audio
        waveform, sr = torchaudio.load(file_path)
        waveform = waveform.squeeze().numpy()

        # pad or truncate
        if len(waveform) > self.max_length:
            waveform = waveform[:self.max_length]
        else:
            waveform = np.pad(waveform, (0, self.max_length - len(waveform)))

        # process
        inputs = self.processor(
            waveform,
            sampling_rate=16000,
            return_tensors='pt',
            padding=False
        )

        label = 1 if row['label'] == 'spoof' else 0

        return {
            'input_values': inputs['input_values'].squeeze(),
            'label': torch.tensor(label, dtype=torch.long)
        }

# test on one sample
dataset_train = ASVspoofDataset(train_df, TRAIN_AUDIO, processor)
sample = dataset_train[0]
print(f"Input shape: {sample['input_values'].shape}")
print(f"Label: {sample['label']}")
print("Dataset class ready ✅")

In [ ]:
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import get_scheduler
from tqdm import tqdm

# Small subset for fine-tuning — 2000 train, 500 dev
train_subset = train_df.sample(2000, random_state=42).reset_index(drop=True)
dev_subset   = dev_df.sample(500, random_state=42).reset_index(drop=True)

train_dataset = ASVspoofDataset(train_subset, TRAIN_AUDIO, processor)
dev_dataset   = ASVspoofDataset(dev_subset,   DEV_AUDIO,   processor)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
dev_loader   = DataLoader(dev_dataset,   batch_size=8, shuffle=False)

# Optimizer
optimizer = AdamW(model.parameters(), lr=1e-4)

# Scheduler
num_epochs = 3
scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=50,
    num_training_steps=num_epochs * len(train_loader)
)

print(f"Train batches: {len(train_loader)}")
print(f"Dev batches:   {len(dev_loader)}")
print("Ready to train ✅")

In [ ]:
from sklearn.metrics import classification_report
import time

def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0
    for batch in tqdm(loader, desc="Training"):
        input_values = batch['input_values'].to(device)
        labels       = batch['label'].to(device)

        outputs = model(input_values=input_values, labels=labels)
        loss    = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
    return total_loss / len(loader)

def eval_epoch(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            input_values = batch['input_values'].to(device)
            labels       = batch['label'].to(device)

            outputs = model(input_values=input_values)
            preds   = outputs.logits.argmax(dim=-1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return all_preds, all_labels

# Training
print("=== Wav2Vec2 Fine-tuning ===\n")
for epoch in range(num_epochs):
    start = time.time()
    loss  = train_epoch(model, train_loader, optimizer, scheduler)
    preds, labels = eval_epoch(model, dev_loader)
    end   = time.time()

    print(f"\nEpoch {epoch+1}/{num_epochs} — Loss: {loss:.4f} — Time: {(end-start):.1f}s")
    print(classification_report(labels, preds, target_names=['bonafide', 'spoof']))

In [ ]:
# Balanced subset — equal bonafide aur spoof
train_bon  = train_df[train_df['label'] == 'bonafide'].sample(1000, random_state=42)
train_sp   = train_df[train_df['label'] == 'spoof'].sample(1000, random_state=42)
train_subset = pd.concat([train_bon, train_sp]).sample(frac=1, random_state=42).reset_index(drop=True)

dev_bon  = dev_df[dev_df['label'] == 'bonafide'].sample(200, random_state=42)
dev_sp   = dev_df[dev_df['label'] == 'spoof'].sample(200, random_state=42)
dev_subset = pd.concat([dev_bon, dev_sp]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Train — Bonafide: {(train_subset['label']=='bonafide').sum()}, Spoof: {(train_subset['label']=='spoof').sum()}")
print(f"Dev   — Bonafide: {(dev_subset['label']=='bonafide').sum()}, Spoof: {(dev_subset['label']=='spoof').sum()}")

# Reset model
model = Wav2Vec2ForSequenceClassification.from_pretrained(
    "facebook/wav2vec2-base", num_labels=2).to(device)

optimizer = AdamW(model.parameters(), lr=1e-4)

train_dataset = ASVspoofDataset(train_subset, TRAIN_AUDIO, processor)
dev_dataset   = ASVspoofDataset(dev_subset,   DEV_AUDIO,   processor)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
dev_loader   = DataLoader(dev_dataset,   batch_size=8, shuffle=False)

scheduler = get_scheduler("linear", optimizer=optimizer,
    num_warmup_steps=50,
    num_training_steps=num_epochs * len(train_loader))

print("Ready ✅")

In [ ]:
print("=== Wav2Vec2 Fine-tuning (Balanced) ===\n")

for epoch in range(num_epochs):
    start = time.time()
    loss  = train_epoch(model, train_loader, optimizer, scheduler)
    preds, labels = eval_epoch(model, dev_loader)
    end   = time.time()

    print(f"\nEpoch {epoch+1}/{num_epochs} — Loss: {loss:.4f} — Time: {(end-start):.1f}s")
    print(classification_report(labels, preds, 
          target_names=['bonafide', 'spoof'], zero_division=0))

In [ ]:
import os

MODEL_SAVE = r'C:\Users\GHANSHYAM\Desktop\voice-clone-detector\results\wav2vec2_finetuned'
os.makedirs(MODEL_SAVE, exist_ok=True)

model.save_pretrained(MODEL_SAVE)
processor.save_pretrained(MODEL_SAVE)

print("Wav2Vec2 saved ✅")

In [ ]:
import torch.nn as nn

class CNNBiLSTM(nn.Module):
    def __init__(self):
        super(CNNBiLSTM, self).__init__()
        
        # CNN — mel spectrogram se spatial features
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        
        # BiLSTM — temporal features
        self.bilstm = nn.LSTM(
            input_size=128 * 16,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.3
        )
        
        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 2)
        )
    
    def forward(self, x):
        # x: (batch, 1, 128, time)
        x = self.cnn(x)                          # (batch, 128, 16, time//8)
        
        b, c, h, t = x.shape
        x = x.permute(0, 3, 1, 2)               # (batch, time, c, h)
        x = x.reshape(b, t, c * h)              # (batch, time, 128*16)
        
        x, _ = self.bilstm(x)                   # (batch, time, 256)
        x = x[:, -1, :]                         # last timestep
        x = self.classifier(x)
        return x

# test
cnn_model = CNNBiLSTM().to(device)
total_params = sum(p.numel() for p in cnn_model.parameters())
print(f"CNN+BiLSTM Parameters: {total_params:,}")
print("Model ready ✅")

In [ ]:
class MelDataset(Dataset):
    def __init__(self, df, audio_dir, n_mels=128, max_len=128):
        self.df = df.reset_index(drop=True)
        self.audio_dir = audio_dir
        self.n_mels = n_mels
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        file_path = f'{self.audio_dir}\\{row["file_id"]}.flac'

        waveform, sr = torchaudio.load(file_path)
        
        # Mel spectrogram
        mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=16000, n_mels=self.n_mels)(waveform)
        mel = torchaudio.transforms.AmplitudeToDB()(mel)

        # pad or truncate time dimension
        if mel.shape[2] > self.max_len:
            mel = mel[:, :, :self.max_len]
        else:
            pad = self.max_len - mel.shape[2]
            mel = torch.nn.functional.pad(mel, (0, pad))

        label = 1 if row['label'] == 'spoof' else 0
        return mel, torch.tensor(label, dtype=torch.long)

# test
mel_dataset = MelDataset(train_subset, TRAIN_AUDIO)
mel, label = mel_dataset[0]
print(f"Mel shape: {mel.shape}")
print(f"Label: {label}")
print("MelDataset ready ✅")

In [ ]:
# DataLoaders
mel_train_dataset = MelDataset(train_subset, TRAIN_AUDIO)
mel_dev_dataset   = MelDataset(dev_subset,   DEV_AUDIO)

mel_train_loader = DataLoader(mel_train_dataset, batch_size=16, shuffle=True)
mel_dev_loader   = DataLoader(mel_dev_dataset,   batch_size=16, shuffle=False)

# Optimizer + Loss
cnn_optimizer = torch.optim.Adam(cnn_model.parameters(), lr=1e-3)
criterion     = nn.CrossEntropyLoss()

# Training
print("=== CNN+BiLSTM Training ===\n")

for epoch in range(5):
    # Train
    cnn_model.train()
    total_loss = 0
    for mels, labels in tqdm(mel_train_loader, desc=f"Epoch {epoch+1}"):
        mels   = mels.to(device)
        labels = labels.to(device)

        outputs = cnn_model(mels)
        loss    = criterion(outputs, labels)

        cnn_optimizer.zero_grad()
        loss.backward()
        cnn_optimizer.step()
        total_loss += loss.item()

    # Eval
    cnn_model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for mels, labels in mel_dev_loader:
            mels    = mels.to(device)
            outputs = cnn_model(mels)
            preds   = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    print(f"\nEpoch {epoch+1}/5 — Loss: {total_loss/len(mel_train_loader):.4f}")
    print(classification_report(all_labels, all_preds,
          target_names=['bonafide', 'spoof'], zero_division=0))

In [ ]:
torch.save(cnn_model.state_dict(), 
    r'C:\Users\GHANSHYAM\Desktop\voice-clone-detector\results\cnn_bilstm.pth')
print("CNN+BiLSTM saved ✅")

In [ ]:
class VoiceAutoencoder(nn.Module):
    def __init__(self, input_dim=61):
        super(VoiceAutoencoder, self).__init__()
        
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU()
        )
        
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, input_dim)
        )
    
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

# Sirf bonafide samples pe train karenge
import numpy as np

SAVE_DIR = r'C:\Users\GHANSHYAM\Desktop\voice-clone-detector\data\processed'

X_train = np.load(f'{SAVE_DIR}\\X_train.npy')
y_train = np.load(f'{SAVE_DIR}\\y_train.npy')

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)

# Sirf bonafide
X_bonafide = X_train_sc[y_train == 0]
print(f"Bonafide samples for autoencoder: {len(X_bonafide)}")

ae_model = VoiceAutoencoder(input_dim=61).to(device)
print(f"Autoencoder params: {sum(p.numel() for p in ae_model.parameters()):,}")
print("Autoencoder ready ✅")

In [ ]:
from torch.utils.data import TensorDataset

# TensorDataset banao
X_bon_tensor = torch.FloatTensor(X_bonafide).to(device)
ae_dataset   = TensorDataset(X_bon_tensor, X_bon_tensor)
ae_loader    = DataLoader(ae_dataset, batch_size=64, shuffle=True)

# Optimizer + Loss
ae_optimizer = torch.optim.Adam(ae_model.parameters(), lr=1e-3)
ae_criterion = nn.MSELoss()

print("=== Autoencoder Training ===\n")

for epoch in range(50):
    ae_model.train()
    total_loss = 0
    for X_batch, _ in ae_loader:
        reconstructed = ae_model(X_batch)
        loss = ae_criterion(reconstructed, X_batch)
        
        ae_optimizer.zero_grad()
        loss.backward()
        ae_optimizer.step()
        total_loss += loss.item()
    
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}/50 — Loss: {total_loss/len(ae_loader):.6f}")

print("\nTraining complete ✅")


In [ ]:
import numpy as np

X_dev = np.load(f'{SAVE_DIR}\\X_dev.npy')
y_dev = np.load(f'{SAVE_DIR}\\y_dev.npy')

scaler_dev = StandardScaler()
X_dev_sc = scaler.transform(X_dev)

# Reconstruction error calculate karo
ae_model.eval()

def get_reconstruction_error(X):
    X_tensor = torch.FloatTensor(X).to(device)
    with torch.no_grad():
        reconstructed = ae_model(X_tensor)
    errors = ((X_tensor - reconstructed) ** 2).mean(dim=1).cpu().numpy()
    return errors

# Bonafide aur spoof ke errors
errors_all = get_reconstruction_error(X_dev_sc)

bon_errors   = errors_all[y_dev == 0]
spoof_errors = errors_all[y_dev == 1]

print(f"Bonafide — Mean error: {bon_errors.mean():.4f}, Std: {bon_errors.std():.4f}")
print(f"Spoof    — Mean error: {spoof_errors.mean():.4f}, Std: {spoof_errors.std():.4f}")

# Threshold — 95th percentile of bonafide errors
tau = np.percentile(bon_errors, 95)
print(f"\nThreshold (tau) at 95th percentile: {tau:.4f}")

# Detection rate
spoof_detected = (spoof_errors > tau).sum()
bon_flagged    = (bon_errors > tau).sum()

print(f"\nSpoof detected:     {spoof_detected}/{len(spoof_errors)} ({spoof_detected/len(spoof_errors)*100:.1f}%)")
print(f"Bonafide flagged:   {bon_flagged}/{len(bon_errors)} ({bon_flagged/len(bon_errors)*100:.1f}%)")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Error distribution
axes[0].hist(bon_errors,   bins=50, alpha=0.7, color='green', label='Bonafide')
axes[0].hist(spoof_errors, bins=50, alpha=0.7, color='red',   label='Spoof')
axes[0].axvline(x=tau, color='black', linestyle='--', linewidth=2, label=f'Threshold={tau:.3f}')
axes[0].set_xlabel('Reconstruction Error')
axes[0].set_ylabel('Count')
axes[0].set_title('Autoencoder — Reconstruction Error Distribution')
axes[0].legend()

# Detection at different thresholds
thresholds = np.percentile(bon_errors, range(80, 100))
spoof_rates = [(spoof_errors > t).mean() for t in thresholds]
fpr_rates   = [(bon_errors > t).mean()   for t in thresholds]

axes[1].plot(fpr_rates, spoof_rates, 'b-o', linewidth=2)
axes[1].set_xlabel('False Positive Rate (Bonafide flagged)')
axes[1].set_ylabel('Spoof Detection Rate')
axes[1].set_title('Autoencoder — Operating Curve')
axes[1].grid(True, alpha=0.3)

for fpr, tpr in zip(fpr_rates, spoof_rates):
    if abs(fpr - 0.05) < 0.01:
        axes[1].annotate(f'FPR=5%\nTPR={tpr:.2f}', 
                        xy=(fpr, tpr), fontsize=10,
                        xytext=(fpr+0.02, tpr-0.05))

plt.tight_layout()
plt.savefig(r'C:\Users\GHANSHYAM\Desktop\voice-clone-detector\results\figures\autoencoder_results.png',
            dpi=150, bbox_inches='tight')
plt.show()

# Save autoencoder
torch.save(ae_model.state_dict(),
    r'C:\Users\GHANSHYAM\Desktop\voice-clone-detector\results\autoencoder.pth')
np.save(r'C:\Users\GHANSHYAM\Desktop\voice-clone-detector\results\tau.npy', 
        np.array([tau]))
print("Autoencoder saved ✅")